# 第29章　モダリティ別・前処理レシピ集

**『医療診断支援AI開発　第1巻　入門編 ― 動かして、自分で作る（第1巻　入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-intro

## 29.1　CT ― HU値とウィンドウ処理

In [ ]:
import numpy as np

def ct_window(hu, center, width):
    low, high = center - width/2, center + width/2
    x = np.clip(hu, low, high)
    return (x - low) / (high - low)          # 0〜1へ

# 代表的なウィンドウ（目的の臓器で使い分ける）
abdomen = ct_window(hu, center=40,  width=400)   # 腹部軟部（肝・膵）
lung    = ct_window(hu, center=-600, width=1500)  # 肺野
bone    = ct_window(hu, center=400,  width=1800)  # 骨（施設・目的で幅がある。中心300〜600/幅1500〜2800程度）
brain   = ct_window(hu, center=40,  width=80)     # 脳

## 実データのDICOMでよくある前処理の分岐

In [ ]:
import numpy as np, pydicom

def load_dicom_hu(path):
    """CT専用。第19.1節・第15章の読込関数と同じ約束：確認できない入力は黙って通さず、例外で止める。"""
    ds = pydicom.dcmread(path)
    if getattr(ds, "Modality", "") != "CT":
        raise ValueError("CT以外のモダリティ。HUは定義されない（29.2の別経路へ）")
    if int(getattr(ds, "NumberOfFrames", 1)) != 1:
        raise ValueError("マルチフレーム。別の変換経路を確認してから扱う")
    if "RescaleSlope" not in ds or "RescaleIntercept" not in ds:
        raise ValueError("HUへの変換係数がない。既定値で埋めず、変換不能として扱う")
    # 本例ではHUを確認する経路を限定する。拒否された画像が非HU・規格違反とは限らない。
    rescale_type = str(getattr(ds, "RescaleType", "")).strip().upper()
    image_type = list(getattr(ds, "ImageType", []))
    multi_energy = str(ds[(0x0018, 0x9361)].value).strip() if (0x0018, 0x9361) in ds else "NO"
    hu_by_image_type = (len(image_type) >= 3 and image_type[0] == "ORIGINAL"
                        and image_type[2] != "LOCALIZER" and multi_energy == "NO")
    if rescale_type != "HU" and not (not rescale_type and hu_by_image_type):
        raise ValueError("変換後の単位がHUと確認できない。別の変換経路で扱う")
    arr = ds.pixel_array.astype(np.float32)
    # HU変換は「格納値 × slope + intercept」だけ。PhotometricInterpretation（MONOCHROME1/2）は
    # 表示の白黒方向を示す情報であって、物理量への変換の前に格納値を書き換える根拠ではない。
    # ここで格納値を反転すると、正しいHUが別の値に変わる。表示極性はX線の表示用経路（29.4）で扱う。
    hu = arr * float(ds.RescaleSlope) + float(ds.RescaleIntercept)   # 生画素→HU（第19章）
    return hu, ds

In [ ]:
import pydicom

pairs = [(p, pydicom.dcmread(p)) for p in paths]
pairs.sort(key=lambda t: float(t[1].ImagePositionPatient[2]))  # z座標で並べる（名前でなく。アキシャル収集のCTが前提）
# 生の格納画素値のまま積むとHUにならない。必ず上の load_dicom_hu を通す
volume = np.stack([load_dicom_hu(p)[0] for p, _ in pairs])   # 正しい頭尾方向のHUボリューム

## 29.2　MRI ― 相対値のzスコア正規化

In [ ]:
def mri_zscore(vol, mask=None):
    region = vol[mask > 0] if mask is not None else vol[vol > 0]  # 背景を除く
    mean, std = region.mean(), region.std()
    return (vol - mean) / (std + 1e-8)

## 29.3　眼底写真 ― CLAHEと余白の切り落とし

In [ ]:
import cv2, numpy as np

def fundus_prep(bgr):
    # ① 眼底の円形領域に外接する矩形で切り出す（周囲の黒い余白は情報を持たず、位置ずれの原因になる）
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    mask = gray > 10                                    # 黒背景と眼底を分ける
    ys, xs = np.where(mask)
    bgr = bgr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]   # 外接矩形で切り出す

    # ② 明度チャネルにCLAHE（局所コントラストを整える）
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[..., 0] = clahe.apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

## 29.4　X線 ― 正規化と、アスペクト比

In [ ]:
import pydicom
import numpy as np
try:                                     # pydicom 3.0 以降の置き場所
    from pydicom.pixels import apply_modality_lut, apply_voi_lut
except ImportError:                      # 2.x（付属の requirements.txt は pydicom 2.4.4）
    from pydicom.pixel_data_handlers.util import apply_modality_lut, apply_voi_lut

# ① 正攻法：装置が意図した表示範囲（VOI変換）を使う。Modality LUT → VOI変換 → 表示極性、の順
arr = apply_modality_lut(ds.pixel_array, ds)
has_voi = ("VOILUTSequence" in ds) or ("WindowCenter" in ds and "WindowWidth" in ds)
if has_voi:                                               # LUT表か、Center/Width の窓か、どちらでも尊重する
    arr = apply_voi_lut(arr, ds)
    xr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
else:
    # ② VOI LUTがないとき：パーセンタイルでクリップしてから伸縮（外れ値1画素に壊されない）
    lo, hi = np.percentile(arr, [0.5, 99.5])
    xr = np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)

# ③ 表示極性は、①②どちらの経路でも最後に共通でそろえる（分岐の内側に置くと、VOIのない画像で漏れる）
if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":   # 値が小さいほど白
    xr = 1.0 - xr                                         # 表示用の白黒方向をそろえる（CTのHU変換ではやらない）

# ④ 最後の手段：素朴な min-max（焼き込み文字や飽和画素があると崩れる）
# xr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)   # 0〜1へ
# 拡張は回転・平行移動・明るさなど、臨床的に許されるものに限る